In [ ]:
%%bash

curl -O https://raw.githubusercontent.com/fpesceKU/BLOCKING/main/block_tools.py &> /dev/null
curl -O https://raw.githubusercontent.com/fpesceKU/BLOCKING/main/main.py &> /dev/null

In [ ]:
import mdtraj as md
import numpy as np 
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm
from scipy.optimize import least_squares
from matplotlib.colors import ListedColormap
from matplotlib.colors import LinearSegmentedColormap
from main import BlockAnalysis
def autoblock(cv, multi=1):
    block = BlockAnalysis(cv, multi=multi)
    block.SEM()
    return block.av, block.sem, block.bs

hexcolors = [
    '#FFAABB',  # pink
    '#EE8866',  # orange
    '#EEDD88',  # light yellow
    '#44BB99',  # mint
    '#99DDFF',  # light cyan
    '#77AADD',  # light blue
    '#DDDDDD'   # pale grey
]

colornames = [
    'pink',
    'orange',
    'light yellow',
    'mint',
    'light cyan',
    'light blue',
    'pale grey'
]

rnb_colors = dict(zip(colornames, hexcolors))

rnb_cmap = ListedColormap(hexcolors, name="rnb")

rnb_cmap_cont = LinearSegmentedColormap.from_list(
    "rnb_continuous",
    hexcolors,
    N=256  # smoothness
)
    
def plotCvsTime(ax,hpro,label,nskip=0,cmap=plt.cm.Blues):
    hpro = np.load(filename)
    lz = hpro.shape[1]
    edges = np.arange(-lz/2,lz/2+1,1)/10.
    dz = (edges[1] - edges[0]) / 2.
    z = edges[:-1] + dz
    print(round(hpro.shape[0]*5e4*0.01*1e-6,1))
    
    hm = np.mean(hpro,axis=0)
    profile = lambda x,a,b,c,d : .5*(a+b)+.5*(b-a)*np.tanh((np.abs(x)-c)/(d*.5)) 
    residuals = lambda params,*args : ( args[1] - profile(args[0], *params) )
    z1 = z[z>0]
    h1 = hm[z>0]
    z2 = z[z<0]
    h2 = hm[z<0]
    p0=[1,1,1,1]
    res1 = least_squares(residuals, x0=p0, args=[z1, h1], bounds=([0]*4,[100]*4)) 
    res2 = least_squares(residuals, x0=p0, args=[z2, h2], bounds=([0]*4,[100]*4))
    thickess = (res1.x[3] + res2.x[3])/2.
    thickess_err = np.abs(res1.x[3] - res2.x[3])
    Lz = (hpro.shape[1]+1)/20
    im = ax.imshow(hpro,extent=[-Lz, Lz,0, hpro.shape[0]*0.5/1000], 
                 cmap=cmap,
                 origin='lower',aspect='auto',norm=LogNorm(vmin=1e-2,vmax=35))
    divider = make_axes_locatable(ax)
    cax = divider.new_horizontal(size="5%", pad=.15)
    f.add_axes(cax)
    cb = f.colorbar(im, cax=cax, orientation="vertical", label=label)
    cax.xaxis.set_label_position('top'); cax.xaxis.set_ticks_position('top')
    ax.set_xlim(-Lz,Lz)
    return hpro, thickess, thickess_err

def plotMap(ax,mat,vmin,vmax,xlabel,ylabel,cmap=plt.cm.bwr,
            ori='vertical',
            cbar=True,cbarlabel='Contact ratio'):
    im = ax.imshow(mat.T,extent=[0, mat.shape[1],0, mat.shape[0]], 
                 cmap=cmap,
                 origin='lower',alpha=1,
                 vmin=vmin,vmax=vmax)
    if cbar:
        divider = make_axes_locatable(ax)
        if ori == 'vertical':
            cax = divider.new_horizontal(size="{:.1f}%".format(5), pad=.1)
        else:
            cax = divider.new_vertical(size="{:.1f}%".format(5), pad=.1)
        f.add_axes(cax)
        cb = f.colorbar(im, cax=cax, orientation=ori,
                    label=cbarlabel)
        cax.xaxis.set_label_position('top'); cax.xaxis.set_ticks_position('top')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

In [ ]:
fasta = """MGSSHHHHHHSSGLVPRGSHMMGDEDWEAEINPHMSSYVPIFEKDRYSGENGDNFNRT
PASSSEMDDGPSRRDHFMKSGFASGRNFGNRDAGECNKRDNTSTMGGFGVGKSFGNRGFSNSRFEDGDS
SGFWRESSNDCEDNPTRNRGFSKRGGYRDGNNSEASGPYRRGGRGSFRGCRGGFGLGSPNNDLDPDECM
QRTGGLFGSRRPVLSGTGNGDTSQSRSGSGSERGGYKGLNEEVITGSGKNSWKSEAEGGES""".replace('\n','')

df_residues = pd.read_csv('scripts/input/residues_CALVADOS2.csv',index_col=0)

mw = round(df_residues.loc[list(fasta)].MW.sum()+18,2)
print(mw)

In [ ]:
f, (ax1,ax2,ax3) = plt.subplots(1, 3, sharex=False, sharey=False, figsize=(7,2.5))
plt.rcParams.update({ 'font.size': 8 })

thickess = []
thickess_err = []
c_dense = []
c_dense_err = []

for name,title,ax in zip(['Ddx4N_66','Ddx4N_216','Ddx4N_366'],
                         ['$c_s=66$ mM','$c_s=216$ mM','$c_s=366$ mM'],[ax1,ax2,ax3]):

    filename = f'data/{name}_{name}_profile.npy'
    _, th, th_err = plotCvsTime(ax,filename,'Concentration  (mM)',nskip=0)
    thickess.append(th)
    thickess_err.append(th_err)
    ax.set_ylabel('Time  (µs)',labelpad=0)
    ax.set_xlabel('$z$  (nm)')
    ax.set_yticks([0,2,4,6,8,10])
    ax.set_title(title,fontsize=8)
    ax.set_ylim(0,10.8)
    df_prop = pd.read_csv(f'data/{name}_ps_results.csv',index_col=0)
    c_dense.append(df_prop.c_dense.values[0]*mw*1e-6)
    c_dense_err.append(df_prop.c_dense_err.values[0]*mw*1e-6)
    
for ax, label in zip((ax1,ax2,ax3), "abc"):
    ax.text(
        -0.33, .99, label,
        transform=ax.transAxes,
        fontsize=12,
        va='bottom',
        ha='right',
        fontweight='bold'
    )

plt.tight_layout(h_pad=2)
plt.savefig('conc_time.pdf', bbox_inches='tight', pad_inches=.05)

In [ ]:
f, (ax1,ax2,ax3) = plt.subplots(1, 3, sharex=False, sharey=False, figsize=(7,2.5))
plt.rcParams.update({ 'font.size': 8 })

for name,c in zip(['Ddx4N_66','Ddx4N_216','Ddx4N_366','FRGG_366'],['light blue','orange','mint']):
    z,h = np.load(f'data/{name}_profiles.npy')
    ax1.plot(z,h,label=str(name.split('_')[-1]),color=rnb_colors[c])
ax1.set_ylabel('Concentration (mM)')
ax1.set_xlim(-150,150)
ax1.legend(frameon=False,title='$c_s$ (mM)',handlelength=.7,loc='upper left',borderpad=0)
ax1.set_xlabel('$z$  (nm)')
    
for i,c in enumerate(['light blue','orange','mint']):
    ax2.errorbar([66,216,366][i],c_dense[i],yerr=c_dense_err[i],marker='o',
         ms=6,lw=0,elinewidth=.4,capsize=1.2,capthick=.4,
         color=rnb_colors[c])
    
ax2.set_xticks([66,216,366])
ax2.set_xlabel('$c_s$  (mM)')
ax2.set_ylabel('Mass concentration (g/mL)')

#ax2.errorbar(exp_cs,exp_conc,yerr=exp_conc_err,marker='s',
#         ms=5,lw=0,elinewidth=.4,capsize=1.2,capthick=.4,
#         color='k')

for i,c in enumerate(['light blue','orange','mint']):
    ax3.errorbar([66,216,366][i],thickess[i],yerr=thickess_err[i],marker='o',
         ms=6,lw=0,elinewidth=.4,capsize=1.2,capthick=.4,
         color=rnb_colors[c])

ax3.set_xticks([66,216,366])
ax3.set_xlabel('$c_s$  (mM)')
ax3.set_ylabel('Interface thickness (nm)')

for ax, label in zip((ax1,ax2,ax3), "abc"):
    ax.text(
        -0.3, .95, label,
        transform=ax.transAxes,
        fontsize=12,
        va='bottom',
        ha='right',
        fontweight='bold'
    )
    
plt.tight_layout()
plt.savefig('conc_profiles_thickness.pdf')

In [ ]:
plt.rcParams.update({'font.size': 10})
f, axs = plt.subplots(2, 3, sharex=False, sharey=False, figsize=(10.5, 6))
axs = axs.ravel()

# ---------- whole maps ----------
for ax, cs, lab in zip(axs[:2], [216, 366], ['a', 'b']):
    cs0 = 66
    cmap = np.load(f'data/Ddx4N_{cs0}_Ddx4N_{cs0}_Ddx4N_{cs0}_homotypic_cmap.npy')
    dmap_2 = pd.DataFrame(data=cmap, index=np.arange(len(fasta))+1, columns=np.arange(len(fasta))+1)
    cmap = np.load(f'data/Ddx4N_{cs}_Ddx4N_{cs}_Ddx4N_{cs}_homotypic_cmap.npy')
    dmap_1 = pd.DataFrame(data=cmap, index=np.arange(len(fasta))+1, columns=np.arange(len(fasta))+1)
    dmap = dmap_1 / dmap_2

    plotMap(ax, dmap, .2, 1.1, xlabel='Residue in central chain',
            ylabel='Residue in surrounding chains', cmap=rnb_cmap_cont,
            ori='vertical', cbar=True)

    ax.set_xticks(np.asarray([0,dmap.index.size//4,dmap.index.size//2,3*dmap.index.size//4,dmap.index.size-1])+.5)
    ax.set_xticklabels([dmap.index.min(), dmap.index[dmap.index.size//4],
                        dmap.index[dmap.index.size//2], dmap.index[3*dmap.index.size//4],
                        dmap.index.max()])
    ax.set_yticks(np.asarray([0,dmap.index.size//4,dmap.index.size//2,3*dmap.index.size//4,dmap.index.size-1])+.5)
    ax.set_yticklabels([dmap.index.min(), dmap.index[dmap.index.size//4],
                        dmap.index[dmap.index.size//2], dmap.index[3*dmap.index.size//4],
                        dmap.index.max()])
    ax.set_title(f'$c_s={cs}$ mM / $c_s=66$ mM')
    ax.annotate(lab, (-.37, 1.04), xycoords='axes fraction', fontsize=14, fontweight='bold')

# ---------- aromatic maps ----------
aa_sel = ['W','Y','F']
aa_mask = [True if aa in aa_sel else False for aa in fasta]

for ax, cs, lab in zip(axs[3:5], [216, 366], ['c', 'd']):
    cs0 = 66
    cmap = np.load(f'data/Ddx4N_{cs0}_Ddx4N_{cs0}_Ddx4N_{cs0}_homotypic_cmap.npy')
    dmap_2 = pd.DataFrame(data=cmap, index=np.arange(len(fasta))+1, columns=np.arange(len(fasta))+1)
    cmap = np.load(f'data/Ddx4N_{cs}_Ddx4N_{cs}_Ddx4N_{cs}_homotypic_cmap.npy')
    dmap_1 = pd.DataFrame(data=cmap, index=np.arange(len(fasta))+1, columns=np.arange(len(fasta))+1)
    dmap = dmap_1 / dmap_2
    dmap = dmap.loc[np.where(aa_mask)[0]+1, np.where(aa_mask)[0]+1]

    plotMap(ax, dmap, .2, 1.1, xlabel='Aromatics in central chain',
            ylabel='Aromatics in surrounding chains', cmap=rnb_cmap_cont,
            ori='vertical', cbar=True)

    ax.set_xticks(np.asarray([0,dmap.index.size//4,dmap.index.size//2,3*dmap.index.size//4,dmap.index.size-1])+.5)
    ax.set_xticklabels([dmap.index.min(), dmap.index[dmap.index.size//4],
                        dmap.index[dmap.index.size//2], dmap.index[3*dmap.index.size//4],
                        dmap.index.max()])
    ax.set_yticks(np.asarray([0,dmap.index.size//4,dmap.index.size//2,3*dmap.index.size//4,dmap.index.size-1])+.5)
    ax.set_yticklabels([dmap.index.min(), dmap.index[dmap.index.size//4],
                        dmap.index[dmap.index.size//2], dmap.index[3*dmap.index.size//4],
                        dmap.index.max()])
    ax.set_title(f'$c_s={cs}$ mM / $c_s=66$ mM')
    ax.annotate(lab, (-.37, 1.04), xycoords='axes fraction', fontsize=14, fontweight='bold')

# ---------- histograms ----------
ax1, ax2 = axs[2], axs[5]

aa_sel = ['F', 'Y', 'W']
aa_mask = np.array([aa in aa_sel for aa in fasta])
aa_mask_1 = np.array([aa in ['E', 'D', 'K'] for aa in fasta])
aa_mask_2 = np.array([aa in ['K'] for aa in fasta])

cs_list = [66, 216, 366]
colors = [rnb_colors['light blue'], rnb_colors['orange'], rnb_colors['mint']]
labels = ['66', '216', '366']

bin_width = 0.1
bins = np.arange(0.35, 2.25, bin_width)
x = bins[:-1] + bin_width / 2.

for cs, color, label in zip(cs_list, colors, labels):
    cmap = np.load(f'data/Ddx4N_{cs}_Ddx4N_{cs}_Ddx4N_{cs}_homotypic_cmap.npy')
    dmap = pd.DataFrame(data=cmap, index=np.arange(len(fasta))+1, columns=np.arange(len(fasta))+1)
    dmap /= dmap.values.mean()
    vals = dmap.loc[np.where(aa_mask_1)[0]+1, np.where(aa_mask_2)[0]+1].values.flatten()

    h = np.histogram(vals, bins=bins, density=False)[0]
    sel = h > 0
    ax1.plot(x[sel], h[sel], color=color, marker='o',
                 lw=1, ms=3, label=label)

ax1.set_title('E/D–K pairs')
ax1.set_xlabel(r'Normalized pair contact, $c_{ij}$ / $\langle c \rangle$')
ax1.set_ylabel('Number of residue pairs')
ax1.annotate('e', (-.20, 1.035), xycoords='axes fraction', fontsize=14, fontweight='bold')


for cs, color, label in zip(cs_list, colors, labels):
    cmap = np.load(f'data/Ddx4N_{cs}_Ddx4N_{cs}_Ddx4N_{cs}_homotypic_cmap.npy')
    dmap = pd.DataFrame(data=cmap, index=np.arange(len(fasta))+1, columns=np.arange(len(fasta))+1)
    dmap /= dmap.values.mean()
    vals = dmap.loc[np.where(aa_mask)[0]+1, np.where(aa_mask)[0]+1].values.flatten()

    h = np.histogram(vals, bins=bins, density=False)[0]
    sel = h > 0
    ax2.plot(x[sel], h[sel], color=color, marker='o',
                 lw=1, ms=3, label=label)

ax2.set_title('Aromatic–aromatic pairs')
ax2.set_xlabel(r'Normalized pair contact, $c_{ij}$ / $\langle c \rangle$')
ax2.set_ylabel('Number of residue pairs')
ax2.legend(title=r'$c_s$ (mM)', frameon=False)
ax2.annotate('f', (-.2, 1.035), xycoords='axes fraction', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('combined_contact_maps.pdf')